# 05 · Recurrentes propios (Cisco)

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "data.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Ejecutar desde el repositorio DL-LAB09")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display, Image, Markdown
from src import data as D, metrics as M, backtest as B
RUN_TRAIN = False  # Cambiar a True solo para generar artefactos que falten.


Tres estrategias: LSTM de un paso con realimentación, LSTM de 24 salidas y seq2seq con calendario y teacher forcing solo en train. Se comparan dos tamaños de estado en validación. El código está en `src/models/rnn.py`; no se usan targets futuros en inferencia.

In [2]:
if RUN_TRAIN:
    import subprocess
    subprocess.run([sys.executable,"-m","src.run_cisco","--stage","rnn"],cwd=ROOT,check=True)

In [3]:
names=['LSTM_AR', 'LSTM_Multi', 'Seq2Seq']
rows=[]
for name in names:
    path=ROOT / "outputs" / "results" / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}: ejecutar la etapa correspondiente")
    rows.append(json.loads(path.read_text()))
display(pd.DataFrame(rows).set_index("modelo"))

,hardware,inference_s,familia,L,estrategia,covariables,params_total,params_entrenables,train_s,val_MASE,...,epochs,max_batches_per_epoch,MAE,RMSE,WAPE,MAPE,MASE,MASE_navidad,train_s_search_total,train_s_criterio
modelo,,,,,,,,,,,,,,,,,,,,,
LSTM_AR,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),9.774532,Recurrentes,168,recursiva,True,11192,11192,12.538953,1.957308,...,6,50,156.283965,1165.874665,0.199970,0.296029,2.004395,2.506769,15.443048,ajuste del candidato seleccionado + refit; bús...
LSTM_Multi,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),13.898156,Recurrentes,168,multi-salida,True,22728,22728,13.334046,1.501184,...,6,50,113.257427,884.602166,0.144917,0.241173,1.452457,2.132335,17.029608,ajuste del candidato seleccionado + refit; bús...
Seq2Seq,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),8.254941,Recurrentes,168,seq2seq,True,22321,22321,16.664309,1.609648,...,5,50,121.795451,937.635889,0.155841,0.250753,1.630471,2.292425,22.013292,ajuste del candidato seleccionado + refit; bús...


In [4]:
for family in ["LSTM_AR","LSTM_Multi","Seq2Seq"]:
    info=json.loads((ROOT/"outputs"/"runs"/family/"selection.json").read_text())
    display(Markdown(f"### {family}: selección solo con validación"))
    display(pd.read_csv(ROOT/"outputs"/"runs"/info["nombre"]/"history.csv"))

### LSTM_AR: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.747382,0.623058,277.933868,2.539513,0.763865
1,2,0.314764,0.544363,258.965332,2.284381,0.824097
2,3,0.227164,0.540356,273.767181,2.309795,0.864562
3,4,0.171250,0.511186,268.113007,2.265050,0.955423
4,5,0.152551,0.618484,315.189240,2.697163,0.916615
5,6,0.161443,0.404529,186.651611,1.957308,0.901407


### LSTM_Multi: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.712861,0.534755,220.516647,2.237817,0.957011
1,2,0.429471,0.369660,194.273575,1.836217,0.999323
2,3,0.314177,0.294902,153.121078,1.601195,0.938274
3,4,0.284595,0.271758,150.836273,1.572684,0.963486
4,5,0.271115,0.261374,146.783966,1.535528,0.999891
5,6,0.250703,0.250209,142.638214,1.501184,1.029951


### Seq2Seq: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.720378,0.577916,265.300140,2.394574,1.221237
1,2,0.367004,0.494713,225.300247,2.125191,1.335570
2,3,0.261839,0.373356,177.032867,1.807576,1.255091
3,4,0.231196,0.349469,170.271820,1.796038,1.277331
4,5,0.209604,0.297820,152.100128,1.609648,1.275527
5,6,0.182804,0.305597,160.716003,1.676103,1.321866


El límite de batches se registra y se vuelve a muestrear con seed por epoch. Stride=1 en ventanas, 150 clientes y 92 orígenes al evaluar. Resultados modestos con este presupuesto no demuestran una inferioridad general de las recurrentes.